In [2]:
%load_ext autoreload
%autoreload 2

import os

from pathlib import Path
import xarray as xr
from polsarpro.io import open_netcdf_beam
from polsarpro.dev.io import read_psp_bin
import shutil

# optional import for progress bar
from dask.diagnostics import ProgressBar

# change to your local C-PolSARpro install dir
c_psp_dir = "/home/c_psp/Soft/bin/"
os.environ["PATH"]+=os.pathsep+f"{c_psp_dir}/data_process_sngl/"
os.environ["PATH"]+=os.pathsep+f"{c_psp_dir}/data_convert/"

# change to your data paths
# original dataset
input_alos_data = Path("/data/psp/test_files/SAN_FRANCISCO_ALOS1_slc.nc")

# input files from C
input_test_dir = Path("/data/psp/SAN_FRANCISCO_ALOS1/")

# output files from C
output_test_dir = Path("/data/psp/res/supervised_wishart_c")
if not os.path.isdir(output_test_dir):
    os.mkdir(output_test_dir)

input_png_dir = Path("/data/psp/labels/")

area_file = None
cluster_file = None

# compute large image
# row1, row2 = 0, 18432
# col1, col2 = 0, 1248

# smaller window
# row1, row2 = 8000, 8500
row1, row2 = 8000, 12000
col1, col2 = 200, 1100

In [ ]:
from matplotlib.image import imread
import xarray as xr
cnt = 0
for f in input_png_dir.glob("*.png"):
    if cnt == 0:
        im = (cnt+1) * (imread(f)[...,0]>0).astype(int)
    else:
        im += (cnt+1) * (imread(f)[...,0]>0).astype(int) 
    cnt += 1

labels = xr.DataArray(im, dims=("y", "x")).chunk({})
labels.to_netcdf("../../data/sample_labels_sf_alos1.nc")

In [ ]:
from scipy.ndimage import label

lab, nc = label(im.data)

from polsarpro.util import S_to_T3 
from polsarpro.io import open_netcdf_beam
from polsarpro.classification import _update_wishart_class_centers

S = open_netcdf_beam(input_alos_data)
T3 = S_to_T3(S)

c = _update_wishart_class_centers(T3, lab, nclass=nc)

In [ ]:
input_dict = dict(
    id=input_test_dir,
    od=output_test_dir,
    iodf="S2",
    af=area_file,
    nwr=7,
    nwc=7,
    ofr=row1,  # offsets are applied to input data and HAA results
    ofc=col1,
    fnr=row2 - row1,
    fnc=col2 - col1,
    cf=cluster_file,
    bmp=0,
)

# making input string: -param1 value1 -param2 value2 etc
param_str = " ".join([f"-{k} {input_dict[k]}" for k in input_dict])
cmd = f"wishart_supervised_classifier.exe {param_str}"
# print(cmd)